# Mini-Project 4: Rainfall Pattern & Crop Suitability Analyser

**Scenario:** Compare illustrative monthly rainfall in Kampala, Gulu, and Mbarara and develop a transparent crop-suitability screening tool.

## Objectives
1. Implement `Region` and `CropRule` classes.
2. Classify rainfall suitability for maize, beans, and coffee by month and region.
3. Implement and verify cosine similarity; calculate cosine similarity, Pearson correlation, and Euclidean distance matrices.
4. Detect rainfall peaks with `scipy.signal.find_peaks` and discuss unimodal/bimodal classification limitations.
5. Plot rainfall lines and a crop-suitability heatmap.
6. Give a short, cautious advisory note for one region.
7. Include an extension-style illustration of year-to-year variability using seeded synthetic rainfall, clearly distinguished from real climate observations.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine as scipy_cosine_distance

SEED = 2026
MONTHS = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

RAINFALL_MM = {
    "Kampala": np.array([120, 140, 180, 200, 220, 180, 90, 70, 60, 100, 110, 130], dtype=float),
    "Gulu": np.array([8, 25, 75, 160, 190, 145, 170, 215, 175, 150, 60, 15], dtype=float),
    "Mbarara": np.array([70, 85, 120, 140, 90, 25, 20, 55, 100, 125, 120, 90], dtype=float),
}
print("Regions:", ", ".join(RAINFALL_MM))
print("Months per region:", len(MONTHS))

## 1. `Region` class and descriptive statistics

The coefficient of variation is calculated as population standard deviation divided by mean rainfall. It describes relative month-to-month variability within the 12 supplied values; it is not a multi-year climate variability estimate.

In [ ]:
@dataclass
class Region:
    """Store a region's 12 monthly rainfall totals and calculate summary measures."""
    name: str
    rainfall_mm: np.ndarray

    def __post_init__(self) -> None:
        self.rainfall_mm = np.asarray(self.rainfall_mm, dtype=float)
        if self.rainfall_mm.ndim != 1 or len(self.rainfall_mm) != 12:
            raise ValueError("rainfall_mm must be a one-dimensional array of 12 months")
        if not np.all(np.isfinite(self.rainfall_mm)) or np.any(self.rainfall_mm < 0):
            raise ValueError("rainfall values must be finite and non-negative")

    def annual_total(self) -> float:
        """Return total rainfall over the 12 supplied months, in mm."""
        return float(np.sum(self.rainfall_mm))

    def monthly_mean(self) -> float:
        """Return average monthly rainfall, in mm/month."""
        return float(np.mean(self.rainfall_mm))

    def wettest_month(self) -> tuple[str, float]:
        """Return (month, rainfall_mm) for the wettest month."""
        i = int(np.argmax(self.rainfall_mm))
        return str(MONTHS[i]), float(self.rainfall_mm[i])

    def driest_month(self) -> tuple[str, float]:
        """Return (month, rainfall_mm) for the driest month."""
        i = int(np.argmin(self.rainfall_mm))
        return str(MONTHS[i]), float(self.rainfall_mm[i])

    def coefficient_of_variation(self) -> float:
        """Return population standard deviation divided by mean, as a percentage."""
        avg = self.monthly_mean()
        return float(np.std(self.rainfall_mm, ddof=0) / avg * 100) if avg else 0.0

    def __repr__(self) -> str:
        return f"Region(name={self.name!r}, annual_total_mm={self.annual_total():.1f})"

regions = {name: Region(name, values) for name, values in RAINFALL_MM.items()}
for region in regions.values():
    print(
        f"{region!r} | mean={region.monthly_mean():.1f} mm | "
        f"wettest={region.wettest_month()} | driest={region.driest_month()} | "
        f"CV={region.coefficient_of_variation():.1f}%"
    )

## 2. Crop rules and monthly classification

The task requests at least three crops. The ranges below are illustrative demonstration rules, not verified agronomic recommendations. A monthly rainfall total alone cannot capture rainfall distribution within a month, soil water storage, drainage, evapotranspiration, temperature, crop variety, or planting date.

Interpretation used here:
- below the crop's minimum: `Drought risk`
- within the range: `Suitable (rainfall screen)`
- above the maximum: `Excess rainfall risk`

For a final academic submission, cite an authoritative agronomy source (for example, a national agricultural research institute, FAO crop guidance, or a peer-reviewed crop-water study) and justify or replace each crop's thresholds using that source.

In [ ]:
@dataclass(frozen=True)
class CropRule:
    """A provisional rainfall screening range for a crop, in mm per month."""
    crop_name: str
    min_monthly_mm: float
    max_monthly_mm: float
    source_note: str = "Illustrative threshold; replace with a cited agronomy source."

    def __post_init__(self) -> None:
        if self.min_monthly_mm < 0 or self.max_monthly_mm <= self.min_monthly_mm:
            raise ValueError("Crop rainfall range must be non-negative and increasing")

    def classify(self, rainfall_mm: float) -> str:
        """Classify one month's rainfall against the crop's provisional range."""
        if rainfall_mm < 0:
            raise ValueError("rainfall cannot be negative")
        if rainfall_mm < self.min_monthly_mm:
            return "Drought risk"
        if rainfall_mm > self.max_monthly_mm:
            return "Excess rainfall risk"
        return "Suitable (rainfall screen)"

crop_rules = [
    CropRule("Maize", 50, 150),
    CropRule("Beans", 40, 120),
    CropRule("Coffee", 60, 180),
]

print("Provisional screening rules:")
for rule in crop_rules:
    print(f"- {rule.crop_name}: {rule.min_monthly_mm}–{rule.max_monthly_mm} mm/month — {rule.source_note}")

In [ ]:
# Long-form classification table: every month × every region × every crop.
classification_rows = []
for region_name, region in regions.items():
    for month, rainfall in zip(MONTHS, region.rainfall_mm):
        for rule in crop_rules:
            classification_rows.append({
                "Region": region_name,
                "Month": str(month),
                "Rainfall_mm": float(rainfall),
                "Crop": rule.crop_name,
                "Classification": rule.classify(float(rainfall)),
            })

try:
    import pandas as pd
    classification_df = pd.DataFrame(classification_rows)
    display(classification_df)
except ImportError:
    for row in classification_rows:
        print(row)

## 3. Correct cosine similarity and verify with SciPy

Cosine similarity between two non-zero vectors is their dot product divided by the product of their Euclidean norms. It is different from `math.cos()`, which takes an angle in radians and returns the cosine of that angle. SciPy's `scipy.spatial.distance.cosine` returns cosine distance, which equals `1 - cosine similarity`.

In [ ]:
def cosine_similarity(x: Sequence[float], y: Sequence[float]) -> float:
    """Compute cosine similarity between equal-length, non-zero numeric vectors."""
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    if x_arr.ndim != 1 or y_arr.ndim != 1 or x_arr.shape != y_arr.shape:
        raise ValueError("Inputs must be 1D vectors of equal length")
    if not np.all(np.isfinite(x_arr)) or not np.all(np.isfinite(y_arr)):
        raise ValueError("Vectors must contain finite values")
    denominator = np.linalg.norm(x_arr) * np.linalg.norm(y_arr)
    if denominator == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector")
    return float(np.dot(x_arr, y_arr) / denominator)

kampala = RAINFALL_MM["Kampala"]
gulu = RAINFALL_MM["Gulu"]
cos_sim = cosine_similarity(kampala, gulu)
scipy_dist = float(scipy_cosine_distance(kampala, gulu))
print(f"NumPy cosine similarity: {cos_sim:.6f}")
print(f"SciPy cosine distance:   {scipy_dist:.6f}")
print(f"1 - SciPy distance:      {1 - scipy_dist:.6f}")
assert np.isclose(cos_sim, 1 - scipy_dist, atol=1e-12)
print("Verification passed.")

## 4. Pairwise similarity and distance matrices

All matrices compare the 12-month rainfall profiles. Pearson correlation measures linear co-movement after centring; Euclidean distance depends on absolute rainfall magnitudes and units. Cosine similarity is scale-insensitive: if one region's monthly rainfall vector is approximately a constant multiple of another's, their directions are similar even when annual totals differ substantially.

In [ ]:
region_names = list(regions)
X = np.vstack([regions[name].rainfall_mm for name in region_names])

cosine_matrix = np.zeros((len(region_names), len(region_names)))
pearson_matrix = np.zeros_like(cosine_matrix)
euclidean_matrix = np.zeros_like(cosine_matrix)

for i in range(len(region_names)):
    for j in range(len(region_names)):
        cosine_matrix[i, j] = cosine_similarity(X[i], X[j])
        pearson_matrix[i, j] = np.corrcoef(X[i], X[j])[0, 1]
        euclidean_matrix[i, j] = np.linalg.norm(X[i] - X[j])

def show_matrix(matrix: np.ndarray, title: str, fmt: str = ".3f") -> None:
    """Display a labelled square matrix with region names."""
    try:
        import pandas as pd
        display(pd.DataFrame(matrix, index=region_names, columns=region_names).style.format(f"{{:{fmt}}}").set_caption(title))
    except ImportError:
        print(title)
        print(np.array2string(matrix, precision=3))

show_matrix(cosine_matrix, "Cosine similarity (unitless)")
show_matrix(pearson_matrix, "Pearson correlation (unitless)")
show_matrix(euclidean_matrix, "Euclidean distance (mm across 12 months)")

## 5. Detect rainy-season peaks

In [ ]:
peak_results = {}
for name, region in regions.items():
    peaks, properties = find_peaks(region.rainfall_mm, prominence=20, distance=2)
    peak_months = [str(MONTHS[i]) for i in peaks]
    # Operational label based only on detected local peaks.
    if len(peaks) <= 1:
        label = "Unimodal (0–1 detected peaks)"
    else:
        label = "Bimodal/multiple peaks (2+ detected peaks)"
    peak_results[name] = {"peak_indices": peaks, "peak_months": peak_months, "label": label}
    print(f"{name}: peaks={peak_months}; algorithmic label={label}")

print("\nClimate interpretation note:")
print("The supplied values are illustrative single-year profiles, not long-term climate normals.")
print("Uganda has regional rainfall differences and often two rainy seasons in parts of the south,")
print("while some northern areas have a more concentrated rainy season. These values alone cannot")
print("reliably establish a region's climate regime.")

## 6. Visualisations

The first figure compares monthly rainfall. The second uses a numeric suitability score for a heatmap: `-1` drought risk, `0` suitable by the provisional screen, and `1` excess-rainfall risk. Crop names and months are both shown so the heatmap can be interpreted without relying on colour alone.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for name, region in regions.items():
    ax.plot(MONTHS, region.rainfall_mm, marker="o", linewidth=2, label=name)
ax.set_title("Illustrative monthly rainfall by region")
ax.set_xlabel("Month")
ax.set_ylabel("Rainfall (mm)")
ax.grid(True, alpha=0.3)
ax.legend(title="Region")
plt.tight_layout()
plt.show()

# One panel per region, rows are crops and columns are months.
score_map = {"Drought risk": -1, "Suitable (rainfall screen)": 0, "Excess rainfall risk": 1}
fig, axes = plt.subplots(1, len(regions), figsize=(16, 5), sharey=True)
for ax, (region_name, region) in zip(axes, regions.items()):
    matrix = np.array([
        [score_map[rule.classify(float(rain))] for rain in region.rainfall_mm]
        for rule in crop_rules
    ])
    im = ax.imshow(matrix, aspect="auto", vmin=-1, vmax=1, cmap="RdYlGn")
    ax.set_title(region_name)
    ax.set_xticks(np.arange(12), MONTHS, rotation=45)
    ax.set_yticks(np.arange(len(crop_rules)), [r.crop_name for r in crop_rules])
    ax.set_xlabel("Month")
axes[0].set_ylabel("Crop")
cbar = fig.colorbar(im, ax=axes, shrink=0.8, ticks=[-1, 0, 1])
cbar.ax.set_yticklabels(["Drought risk", "Suitable", "Excess rainfall risk"])
fig.suptitle("Monthly crop rainfall-screen classifications (illustrative rules)", y=1.02)
plt.tight_layout()
plt.show()

## 7. Short advisory note for farmers in Gulu 
The supplied Gulu profile has very low rainfall in January–February and December, with higher values from approximately April through October. Under the provisional rainfall screen used here, those dry months may create water-stress concerns, while wetter months may be more suitable for crops whose rainfall needs match the observed totals. However, monthly totals cannot establish exact planting dates. Farmers should use local extension advice, soil moisture, seed variety recommendations

## 8. Extension illustration: synthetic multi-year monthly variability

In [ ]:
rng = np.random.default_rng(SEED)
years = 10
synthetic_by_month = {}

# Create a clearly labelled synthetic example by adding seeded noise to each region's
# illustrative profile. Clip rainfall at zero; this is for demonstrating the boxplot only.
for name, region in regions.items():
    noise = rng.normal(loc=0.0, scale=0.18, size=(years, 12))
    synthetic = np.clip(region.rainfall_mm[None, :] * (1 + noise), 0, None)
    synthetic_by_month[name] = synthetic

fig, axes = plt.subplots(1, len(regions), figsize=(16, 5), sharey=True)
for ax, (name, values) in zip(axes, synthetic_by_month.items()):
    ax.boxplot([values[:, m] for m in range(12)], labels=MONTHS, showfliers=False)
    ax.set_title(f"{name} (synthetic)")
    ax.set_xlabel("Month")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Rainfall (mm)")
fig.suptitle("Demonstration only: synthetic 10-year monthly rainfall variability", y=1.02)
plt.tight_layout()
plt.show()
print("Synthetic values are for code demonstration, not observed climate data.")

## 9. Findings & Limitations

The three illustrative profiles have different annual totals and seasonal patterns. Kampala's supplied values peak in May and are comparatively moderate during the middle of the year, while Gulu has very dry values at the beginning and end of the year and a pronounced wetter period through much of the middle months. Mbarara's supplied profile has a peak around April and another smaller rise around October–November, separated by a notably dry June–August period. These descriptions apply only to the given 12 monthly values; they are not estimates of long-term averages.

The Region class calculates annual totals, means, wettest and driest months, and coefficient of variation. The CropRule class makes the classification logic reusable, but the numerical crop thresholds are provisional teaching assumptions. They must be replaced or justified with cited agronomic sources before the output can be used for farm decisions. Monthly rainfall alone is not enough to determine suitability because soil, drainage, crop variety, rainfall timing, temperature, and evaporation matter.

## 10. Validation checks

These checks cover expected values, similarity verification, and invalid input handling. The accompanying `tests/` folder contains pytest tests for automated assessment.

In [ ]:
# Annual totals can be checked directly from the supplied arrays.
assert regions["Kampala"].annual_total() == float(np.sum(RAINFALL_MM["Kampala"]))
assert regions["Gulu"].wettest_month()[0] == "Aug"
assert np.isclose(cosine_similarity(kampala, gulu), 1 - scipy_cosine_distance(kampala, gulu))

# Invalid inputs must fail clearly.
try:
    Region("Bad", np.array([1, 2, 3]))
    raise AssertionError("Expected invalid rainfall length to raise ValueError")
except ValueError:
    pass

try:
    cosine_similarity([0, 0], [1, 2])
    raise AssertionError("Expected zero vector to raise ValueError")
except ValueError:
    pass

print("All notebook validation checks passed.")